# Petrov–Galerkin MHM on polytopes

Fernando, Martins, Pereira and Valentin (2023), [DOI 10.1007/s40314-023-02304-y](https://doi.org/10.1007/s40314-023-02304-y).

The base field $p=R f-RB\lambda+c$ and enriched field $\widetilde p=p-RB\lambda_R$ are different numerical quantities. In the physical Darcy convention, $\lambda_R=-\alpha K_{\min}[p-g]/(2H_E)$. Only $\lambda+\lambda_R$ is macro conservative. Raw volume gradients are not asserted to be globally H(div).

The explicit choice $\alpha=0.1$ is a selected parameter: the paper does not state its numerical value. The theorem requires sufficiently small alpha and compatible spaces; a residual check does not prove uniform stability.

The main equations execute from source and state their current discretization. The original archived comparisons below execute with `--historical` and require their exact attributed payloads. `--study` runs the complete public current acquisition declared in the catalogue, with its own output provenance.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/aec78d4901f31bcaeee88585f4204f26f28ace8cbc89304f49016e2fa3f01ccb/59_pgmhm-companion.zip"
COMPANION_SHA256 = "aec78d4901f31bcaeee88585f4204f26f28ace8cbc89304f49016e2fa3f01ccb"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/59_pgmhm.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from pymhm.fem.traces.interval import FaceSpace, SkeletonSpace
from pymhm.meshes.triangle import TriangleMesh
from pymhm import assemble
from examples.formulations.darcy import define_darcy
from examples.formulations.penalty import add_jump_form, recover_penalty


In [ ]:
import inspect
from examples.formulations.darcy import local_equations
from examples.formulations.penalty import jump_equation
print(inspect.getsource(local_equations))
print(inspect.getsource(jump_equation))


## Executable nonzero-boundary patch

With $p=1+x^2+2y^2$, $K=I$, the source is $f=-6$ and the exact physical flux is $q=(-2x,-4y)$. The trace has degree one and local pressure degree three.

In [ ]:
mesh = TriangleMesh.unit_square(2)
skeleton = SkeletonSpace(mesh, tuple(FaceSpace.uniform(1) for _ in mesh.faces))
def pressure(x):
    return 1 + x[:, 0]**2 + 2*x[:, 1]**2
definition = define_darcy(
    mesh, skeleton=skeleton, degree=3, source=-6,
    local_refinement=1, quadrature_order=6,
)
base = assemble(definition.problem)
# The jump form acts on the executed local response maps.
system, penalties, lower = add_jump_form(definition, base, dirichlet=pressure, alpha=0.1)
coefficients = system.solve()
result = recover_penalty(definition, system, coefficients, penalties, lower, alpha=0.1)
assert result.l2_error(pressure, enriched=True) < 1e-10
assert max(abs(result.conservation_residuals())) < 1e-10
print("Enriched pressure L2 error:", result.l2_error(pressure, enriched=True))
print("Maximum macro balance:", max(abs(result.conservation_residuals())))


## Archived multilevel evidence

The Section 6.1 pressure is $\sin(2\pi x)\sin(2\pi y)$. Macro refinement and skeletal refinement are separate studies. Triangular macro refinement uses one local element with $k=\ell+2$, as stated in the article. L-shaped connectivity and alpha are explicitly selected; matching historical plotted values is not claimed. Divergence errors below use actual broken derivatives, and the archived standard H(div) norm has weight one.

Exact/base/enriched fields share signed scales. Every spatial panel contains the actual macro edges and independent one-sided samples.

In [ ]:
if HISTORICAL_REPLAY:
    record = json.loads((root / "examples/results/pgmhm/comparison.json").read_text())
    assert record["source_changed_during_run"] is False
    print("Recorded cases:", len(record["rows"]))
    for row in record["rows"]:
        if row["study"] == "macro" and row["level"] == 32:
            print(row["mesh"], row["trace_degree"], row["pressure_l2"], row["flux_l2"],
                  row["enriched_divergence_l2"])


In [ ]:
if HISTORICAL_REPLAY:
    display(Image(filename=str(root / "docs/figures/pgmhm/triangles-convergence.png")))


In [ ]:
if HISTORICAL_REPLAY:
    display(Image(filename=str(root / "docs/figures/pgmhm/L-polygons-convergence.png")))


In [ ]:
if HISTORICAL_REPLAY:
    display(Image(filename=str(root / "docs/figures/pgmhm/skeleton-convergence.png")))


In [ ]:
if HISTORICAL_REPLAY:
    display(Image(filename=str(root / "docs/figures/pgmhm/triangles-fields.png")))


In [ ]:
if HISTORICAL_REPLAY:
    display(Image(filename=str(root / "docs/figures/pgmhm/L-polygons-fields.png")))


Full campaigns run separately from this notebook:

```bash
python -m examples.pgmhm_campaign --workers 8
python -m examples.plot_pgmhm
```

The article's heterogeneous inclusions and SPE10 comparisons require separate refined references; this smooth campaign does not substitute for them.

## Heterogeneous square-annulus inclusions

The Section 6.2 control retains 729 hollow square inclusions, contrast $10^5$,
32 L-shaped macroelements, P2 local pressure and two P0 segments per macroface.
MHM and PGMHM have identical approximation spaces. The radius-to-period ratios
are explicitly scaled from the cited precursor; the historical local meshes
and heterogeneous-case stabilization parameter are unavailable.

The independently assembled DOLFINx/UFL CG2 fields have their own refinement
increments. Each comparison below identifies its numerical denominator, which
must be distinguished from an exact solution. Principal and correction
coefficient arrays are retained separately during physical-field evaluation.
The fixed value $\alpha=0.1$ and the material are not adjusted to match a curve.

The common comparison denominator is the 9,320,809-unknown interface-graded CG2 field. Its own last increments are 0.05513% in pressure, 2.93166% in raw flux and 2.32107% in weighted flux. PGMHM local factors 1/2/4 have flux differences 28.0226%/18.5954%/13.7916%; the last reference increment remains about 21% of the finest measured distance, not an error bound. The nonnested piecewise-uniform versus graded reference difference is integrated separately, without changing any MHM coefficients.


In [ ]:
if HISTORICAL_REPLAY:
    import hashlib

    def checked_json(path):
        """Verify every recorded physical-field digest before replaying numerical norms."""
        row = json.loads(path.read_text())
        for name in ("archive", "reference", "mhm"):
            if name not in row or name + "_sha256" not in row:
                continue
            candidate = path.parent / row[name]
            if not candidate.exists():
                candidate = path.parent.parent / row[name]
            assert hashlib.sha256(candidate.read_bytes()).hexdigest() == row[name + "_sha256"]
        return row

    inclusions = root / "examples/results/pgmhm-inclusions"
    for path in sorted(inclusions.glob("classical-cg2-*.json")):
        row = checked_json(path)
        print(path.name, "DOFs", row["dofs"], "residual", row["residual"])
        if "norms" in row:
            print("Own last increment:", row["norms"][-1])
    for path in sorted(inclusions.glob("mhm-pgmhm-*-comparison.json")):
        row = checked_json(path)
        print(path.name, "reference:", row["reference"], row["rows"][-1])


In [ ]:
if HISTORICAL_REPLAY:
    for name in ("material-profile", "pressure-fields", "physical-norms"):
        display(Image(filename=str(root / "docs/figures/pgmhm-inclusions" / f"{name}.png")))


## SPE10 component identification and resolved controls

The published material image identifies $K_z$ from its spatial colors, while
the printed pressure plateaus support $K_x=K_y$ under the stated zero-source
problem. This observable inconsistency is retained explicitly. The flow
comparison uses the profile-supported $K_x$ configuration, unchanged units,
P2 local pressure, P0 traces and $\alpha=0.1$.

A classical conforming RT2/P2 sequence supplies the physical-field baseline.
Local material fitting and additional trace moments are separate controls:
integration cuts alone do not let a crossing polynomial represent a material
interface gradient jump. All reported fluxes below are raw $-K\nabla p$,
not H(div) reconstructions. The independent reference has a nonzero refinement
increment, which is reported separately from MHM differences.

With the nominal trace fixed, fitting the local P2 spaces reduces the raw-flux distance from 60.8803% to 12.2249%. Keeping those 112,896 fitted triangles fixed and adding P0 trace moments at material intersections reduces it to 5.24665%; the pressure distance is 0.03564%. The reference's own last flux increment is 1.99652%, so these distances are not exact errors or an orthogonal decomposition.

With the same 562 free trace/kernel coordinates, local refinement to 182,976 fitted triangles reduces the raw-flux distance to 4.19206%. Enriching only the trace on these latter local meshes gives 4.18370% (754 free coordinates). Both use the same RT2 240×880 denominator. Its 1.99652% own increment is approximately 48% of that last distance; the remaining discrepancy requires a finer classical baseline for substantially smaller reference uncertainty.


In [ ]:
if HISTORICAL_REPLAY:
    spe = root / "examples/results/pgmhm-spe10/kx"
    for path in sorted(spe.rglob("*-comparison.json")):
        row = checked_json(path)
        print(path.relative_to(spe), "reference:", row["reference"])
        print(row["norms"][-1])
    for name in ("material-profiles", "fields", "norms", "control-flux-components", "resolution-controls"):
        display(Image(filename=str(root / "docs/figures/pgmhm-spe10" / f"{name}.png")))
